# Amazon Bedrock AgentCore Runtime 上の TypeScript MCP サーバーで MCP クライアントをテストする

## 概要

このチュートリアルでは、Amazon Bedrock AgentCore ランタイム環境を使用して TypeScript ベースの MCP（Model Context Protocol）サーバーをホストする方法を学習します。

### チュートリアルの詳細

| 情報               | 詳細                                                       |
|:-------------------|:-----------------------------------------------------------|
| チュートリアルタイプ | TypeScript MCP サーバーのホスティング                      |
| ツールタイプ       | MCP サーバー                                                |
| チュートリアル構成要素 | AgentCore Runtime 上での TypeScript MCP サーバーのホスティング |
| チュートリアル垂直領域 | クロス垂直領域                                              |
| 例の複雑さ         | 簡単                                                        |
| 使用 SDK           | Anthropic の MCP 用 TypeScript SDK                         |


### チュートリアルの概要

1. AgentCore Runtime の認証では、デプロイされた MCP サーバーにアクセスするための JWT トークンを提供するために Amazon Cognito を使用します。

2. MCP サーバーは TypeScript で記述され、[カスタムフローを使用してデプロイ](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html)されます。

3. MCP クライアントは Python で記述されています。
   _注: MCP クライアントは任意の言語で記述できます。_

## 前提条件

このチュートリアルを実行するには、以下が必要です:
- Node.js v22 以降（MCP サーバー用）
- Python 3.10+（MCP クライアント用）
- Docker（コンテナ化用）  
- Docker イメージを保存するための Amazon ECR（Elastic Container Registry）  
- Bedrock AgentCore へのアクセス権を持つ AWS アカウント  
- MCP（Model Context Protocol）ライブラリ
- Docker が実行中であること

In [ ]:
#!uv add -r requirements.txt --active

## MCP（Model Context Protocol）の理解

MCP は、AI モデルが外部データとツールに安全にアクセスできるようにするプロトコルです。主要な概念:

* **Tools（ツール）**: AI がアクションを実行するために呼び出すことができる関数
* **Prompts（プロンプト）**: サーバーが LLM と対話するための構造化されたメッセージと指示を提供できるようにする
* **Streamable HTTP**: AgentCore Runtime で使用されるトランスポートプロトコル
* **Session Isolation（セッション分離）**: 各クライアントは `Mcp-Session-Id` ヘッダーを介して分離されたセッションを取得
* **Stateless Operation（ステートレス操作）**: サーバーはスケーラビリティのためにステートレス操作をサポートする必要がある

AgentCore Runtime は、MCP サーバーがデフォルトパスとして `0.0.0.0:8000/mcp` でホストされることを想定しています。

## ステップ 1: 認証用の Amazon Cognito の設定

AgentCore Runtime には認証が必要です。デプロイされた MCP サーバーにアクセスするための JWT トークンを提供するために、Amazon Cognito を使用します。

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role, setup_cognito_user_pool

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## ステップ 2: IAM 実行ロールの作成

開始する前に、AgentCore Runtime 用の IAM ロールを作成しましょう。このロールは、ランタイムが動作するために必要な権限を提供します。

In [ ]:
tool_name = "mcp_server_ac"
print(f"Creating IAM role for {tool_name}...")
agentcore_iam_role = create_agentcore_role(agent_name=tool_name)
print(f"IAM role created ✓")
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")

## ステップ 3: MCP サーバーの作成

ここで、2つのシンプルなツールと1つのプロンプトを持つ TypeScript MCP サーバーを作成しましょう。このチュートリアルの下にある src フォルダーに移動します。

1. 依存関係のインストール

```
npm install
```

2. AWS 認証情報の設定
```
aws configure
export AWS_ACCESS_KEY_ID=your_access_key
export AWS_SECRET_ACCESS_KEY=your_secret_key
export AWS_REGION=us-east-1
```

3. サーバーの起動（ローカルで実行）
```
npm run start
```

## ステップ 4: Docker による MCP サーバーのデプロイメント

注: これはスターターツールキットなしでエージェントまたは MCP サーバーをデプロイするための手動ステップです

https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

1. ECR リポジトリの作成
```
aws ecr create-repository --repository-name mcp-server --region us-east-1
```
2. ECR へのイメージのビルドとプッシュ
```
# ログイントークンの取得
aws ecr get-login-password --region us-east-1 | \
  docker login --username AWS --password-stdin [account-id].dkr.ecr.us-east-1.amazonaws.com

docker buildx --platform linux/arm64 \
  -t [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest --push .
```

3. Bedrock AgentCore へのデプロイ

    - AWS コンソール → Bedrock → AgentCore → Create Agent に移動
    - プロトコルとして MCP を選択
    - Agent Runtime の設定:
        - Image URI: [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest
        - Bedrock モデルアクセス用の IAM 権限を設定
        - Agent Sandbox でデプロイしてテスト
    - Discovery url について: 上記から URL を選択、cognito_config['discovery_url']
    - Client id について: 上記からクライアント ID を選択、cognito_config['client_id']
    - 実行ロールについて: 上記から ARN を選択、agentcore_iam_role['Role']['Arn']


## ステップ 5: リモートアクセス用の設定の保存

デプロイされた MCP サーバーを呼び出す前に、Agent ARN（ステップ 4 から ARN を取得）と Cognito 設定を AWS Systems Manager Parameter Store と AWS Secrets Manager に保存して、簡単に取得できるようにしましょう:

In [ ]:
import boto3
import json

boto_session = Session()
region = boto_session.region_name

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='mcp_server/cognito/credentials',
        Description='Cognito credentials for MCP server',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials stored in Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='mcp_server/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Cognito credentials updated in Secrets Manager")

 # NOTE: Add your agent arn that you created in Step 4
agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime/agent_arn',
    Value="Add your agent arn that you created in step 4", 
    Type='String',
    Description='Agent ARN for MCP server',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

## ステップ 6: リモートテストクライアントの作成

ここで、デプロイされた MCP サーバーをテストするためのクライアントを作成しましょう。このクライアントは、AWS から必要な認証情報を取得し、デプロイされたサーバーに接続します:

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")
     
        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Error: BEARER_TOKEN not retrieved properly")
        sys.exit(1)
    

    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## ステップ 7: デプロイされた MCP サーバーのテスト

リモートクライアントを使用してデプロイされた MCP サーバーをテストしましょう:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python my_mcp_client_remote.py

## ステップ 8: MCP ツールのリモート呼び出し

ここで、ツールをリストするだけでなく、完全な MCP 機能を実証するためにツールを呼び出す拡張クライアントを作成しましょう:

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Using AWS region: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Retrieved Agent ARN: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Retrieved bearer token from Secrets Manager")
        
    except Exception as e:
        print(f"Error retrieving credentials: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)
                
                try:
                    print("\n➕ Testing add(5, 3)...")
                    add_result = await session.call_tool(
                        name="add",
                        arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                try:
                    print("\n✖️  Testing subtract(10, 2)...")
                    substract_result = await session.call_tool(
                        name="subtract",
                        arguments={"a": 10, "b": 2}
                    )
                    print(f"   Result: {substract_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                print("\n✅ MCP tool testing completed!")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## ツール呼び出しのテスト

実際に MCP ツールを呼び出してテストしましょう:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

## 次のステップ

AgentCore Runtime に MCP サーバーを正常にデプロイしたので、以下を実行できます:

1. **ツールの追加**: 追加のツールで MCP サーバーを拡張する
2. **カスタム認証**: カスタム JWT オーソライザーを実装する
3. **統合**: 他の AgentCore サービスと統合する

# 🎉 おめでとうございます！

以下の作業を正常に完了しました:

✅ **カスタムツールを含む TypeScript MCP サーバーを作成**  
✅ **Amazon Cognito で認証を設定**  
✅ **AgentCore Runtime を使用して AWS にデプロイ**  
✅ **適切な認証でリモート呼び出し**  
✅ **MCP の概念とベストプラクティスを学習**  

MCP サーバーは Amazon Bedrock AgentCore Runtime 上で実行中で、本番環境での使用準備が整いました！